In [1]:
import pandas as pd
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#pip install tensorflow

In [5]:
# Load dataset
df = pd.read_csv('../audio.csv')
df.set_index('File_Name', inplace=True)

# Initialize lists for features and labels
features = []
labels = []
genders = [] 

# Read WAV files and extract features
for f in df.index:
    file_path = '../data/' + f
    try:
        signal, rate = librosa.load(file_path, sr=None)
        df.at[f, 'length'] = len(signal) / rate
        
        # Extract MFCCs
        mfccs = librosa.feature.mfcc(y=signal, sr=rate, n_mfcc=13)
        mfccs_mean = np.mean(mfccs, axis=1)

        if mfccs_mean.shape[0] == 13:
            features.append(mfccs_mean)
            labels.append(df.at[f, 'Emotion'])  
            genders.append(df.at[f, 'Gender'])  

    except Exception as e:
        print(f"Error processing file {file_path}: {e}")

# Convert features and labels to arrays
X = np.array(features)
y = np.array(labels)
genders = np.array(genders)

# Create a DataFrame
df_features = pd.DataFrame(X)
df_features['Emotion'] = y
df_features['Gender'] = genders

# Oversampling process
gender_distribution = df_features.groupby(['Emotion', 'Gender']).size()
max_count = gender_distribution.max()

# Initialize balanced DataFrame
balanced_df = pd.DataFrame()

for (emotion, gender), count in gender_distribution.items():
    label_df = df_features[(df_features['Emotion'] == emotion) & (df_features['Gender'] == gender)]
    if len(label_df) < max_count:
        label_df = label_df.sample(max_count, replace=True)
        
    balanced_df = pd.concat([balanced_df, label_df], ignore_index=True)

# Check the balanced dataset
print("Balanced dataset distribution:")
print(balanced_df['Emotion'].value_counts())
print(balanced_df['Gender'].value_counts())

Balanced dataset distribution:
Emotion
angry        192
calm         192
disgust      192
fearful      192
happy        192
neutral      192
sad          192
surprised    192
Name: count, dtype: int64
Gender
F    768
M    768
Name: count, dtype: int64


In [9]:
# Assuming df_features has been created and contains the MFCC features, Emotion, and Gender
# Prepare features and labels
X = balanced_df.drop(columns=['Emotion', 'Gender']).values
y = balanced_df['Emotion'].values

# Reshape X for CNN input (samples, height, width, channels)
X = X.reshape(X.shape[0], 13, 2, 1)  # 13 MFCCs, 2 for mean and std, 1 for channel

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Define the CNN model
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(13, 2, 1)))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(np.unique(y_encoded)), activation='softmax'))  # Number of classes

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)

# Evaluate the model
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Print evaluation metrics
print("Classification Report:")
print(classification_report(y_test, y_pred_classes))

# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred_classes)

# Plotting the confusion matrix
plt.figure(figsize=(10, 7))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

ValueError: cannot reshape array of size 19968 into shape (1536,13,2,1)

In [10]:
balanced_df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,Emotion,Gender
0,-588.253723,59.247963,-8.007299,4.542206,-1.324075,0.973126,-9.401972,2.055399,-7.919817,-6.678195,-1.995374,-2.865559,-2.086076,angry,F
1,-564.981812,62.465103,-14.487927,3.440038,-1.181023,0.262856,-10.752198,2.646129,-10.449388,-5.059660,-0.900703,-1.512436,-1.119551,angry,F
2,-602.720886,62.845188,-8.394002,6.589411,-4.375909,1.435627,-10.295984,3.848105,-10.856516,-5.512211,-4.194157,-1.542342,-4.706222,angry,F
3,-590.882202,59.846298,-14.253839,4.344372,-5.582291,-0.559232,-13.923412,3.807024,-11.936244,-6.598402,-4.050676,-2.321143,-4.304013,angry,F
4,-477.003082,51.984638,-22.436211,0.079663,-4.303525,-6.375623,-13.338382,3.294466,-10.275780,-8.972215,-0.730880,-4.789858,-4.328777,angry,F
